# Lesson 79 · Music embeddings — letting similar songs meet in vector space: contrastive learning pulls similar styles together and pushes genres apart

**Goal**: train a contrastive music encoder — clips of the same song close together, different songs far apart — that outputs a fixed-length embedding.

> Links back to similarity in L10; be clear whether the vectors come from training in this lesson or from pretrained weights.

🔗 **Aurora connection**: the three implementations in this section already live in `aurora.music.embed` and can be used directly via `from aurora.music import MusicEncoder, triplet_loss, nt_xent_loss` (requires `pip install aurora[music]`). On the retrieval side, `aurora.music.knn_search` will use these vectors for nearest-neighbor queries in L80.

← **Previous lesson** · [L78 · Beat tracking from scratch](L78_beat_tracking.ipynb)

> Last lesson covered **beat tracking from scratch**: the onset envelope, autocorrelation and BPM estimation.
> This lesson explores **music embeddings**.

## This lesson's story: Shazam's little brother — letting an AI judge whether two songs share a style

Shazam identifies songs by fingerprint matching. But ask it "find me 10 pieces in a style like Beethoven's *Moonlight Sonata*" and Shazam can't do it — it has no notion of "similar style".

The idea behind **contrastive learning**: map music clips into a vector space so that "similar music sits close together and different music is pushed apart".

```
Moonlight Sonata, clip 1 ───────┐
Moonlight Sonata, clip 2 ────── → close together in vector space  ← different clips of the same piece
a heavy-metal track (different style) → pushed apart in vector space
```

**Triplet loss**: train on triplets, one at a time:
- **Anchor**: the reference sample
- **Positive**: same class/style as the anchor
- **Negative**: a different class from the anchor

```
L = max(d(a,p) - d(a,n) + margin, 0)
```
- If `d(a,p) < d(a,n) - margin` (the positive is already closer than the negative by margin) → loss=0 (this triplet is done)
- Otherwise → the gradient pushes anchor/positive together and anchor/negative apart

**What the margin is for**: it stops the model from "cheating" — mapping every vector to the same point, which would make loss=0 but tell nothing apart.

**This section's flow**: implement triplet_loss → MusicEncoder (CNN + AdaptiveAvgPool) → L2 normalization → cluster visualization.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import torch
import torch.nn as nn
import numpy as np

### 🤔 Think first: why would "pull together / push apart" teach a model about "style"?

Don't look at the formula yet. Imagine the encoder is only allowed to obey two commands:

- **Two clips of the same song** → move a little closer in vector space;
- **Clips of different songs** → move a little further apart.

Repeat that thousands upon thousands of times, and the only way the encoder can satisfy **all** of these near/far constraints **at once** is to discover for itself "which acoustic features tell songs and styles apart", and encode them in the directions of its vectors. Nobody tells it "what rock is" — it is "forced" out of these near/far relationships.

That is the core intuition of contrastive learning: **don't define the classes, only define "who is like whom", and let the geometry grow its own meaning.** The triplet loss below is these two commands written as a differentiable formula.

## 1. Contrastive learning: the triplet loss

Each training step works on a triplet (anchor, positive, negative):
- `anchor`: the reference clip
- `positive`: another clip from the same song as the anchor
- `negative`: a clip from a different song

**Triplet loss formula**:

```
L = mean( max(0,  d(anchor, pos) - d(anchor, neg) + margin) )
```

Here `d` is the L2 distance and `margin` is a safety margin (typically 0.2). When `d_neg - d_pos > margin` the loss is 0, meaning the negative sample is already far enough away and produces no more gradient.

In [ ]:
# Intuition demo: the loss surface
import torch

margin = 0.2
d_pos_vals = torch.linspace(0, 2, 50)
d_neg_fixed = 1.0  # fixed negative distance

loss_vals = torch.clamp(d_pos_vals - d_neg_fixed + margin, min=0)

print("d_pos\t\td_neg\t\tloss")
for i in [0, 12, 25, 37, 49]:
    print(f"{d_pos_vals[i]:.2f}\t\t{d_neg_fixed:.2f}\t\t{loss_vals[i]:.4f}")

## 2. MusicEncoder: mel → a fixed-length vector

Input: a mel feature matrix of shape `(B, 1, T, n_mels)`, with a variable-length time axis.
Architecture:
```
Conv2d(1→16, k=3) → ReLU → MaxPool(2,2)
Conv2d(16→32, k=3) → ReLU → AdaptiveAvgPool → global flatten
Linear(32*n_mels//4, embed_dim=128)
```
`AdaptiveAvgPool2d((1, n_mels//4))` squeezes the time axis down to 1, so the output length doesn't depend on the input duration — the key to handling variable-length audio.

In [ ]:
class MusicEncoder(nn.Module):
    def __init__(self, n_mels=64, embed_dim=128):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                       # T//2, n_mels//2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, n_mels // 4)),   # 1, n_mels//4
        )
        self.fc = nn.Linear(32 * (n_mels // 4), embed_dim)

    def forward(self, x):                             # x: (B,1,T,n_mels)
        h = self.cnn(x)                               # (B,32,1,n_mels//4)
        h = h.flatten(1)                              # (B, 32*n_mels//4)
        return self.fc(h)                             # (B, embed_dim)

# Check input and output shapes
enc = MusicEncoder(n_mels=64, embed_dim=128)
dummy = torch.randn(4, 1, 100, 64)   # batch=4, T=100 frames, n_mels=64
out = enc(dummy)
print(f"Input shape: {dummy.shape}  →  output shape: {out.shape}")
assert out.shape == (4, 128), "wrong shape"
print("✅ MusicEncoder shape check passed")

## 3. L2 normalization: putting the vectors on the unit hypersphere

After training, L2-normalize the output:

```
v_norm = v / ||v||_2
```

After normalization every vector has length 1, and then:
- **Cosine similarity** = `v_a · v_b` (a plain dot product, no need to divide by the lengths)
- L2 distance and cosine distance are monotonically equivalent: `||a-b||^2 = 2 - 2*(a·b)`

This lets ANN retrieval (a FAISS inner-product index) stand in directly for cosine search, and it's faster.

In [ ]:
def l2_normalize(x: torch.Tensor) -> torch.Tensor:
    """L2-normalize along the last dimension."""
    return x / (x.norm(dim=-1, keepdim=True) + 1e-8)

v = torch.randn(3, 128)
v_n = l2_normalize(v)
norms = v_n.norm(dim=-1)
print(f"Length of each vector after normalization: {norms.tolist()}")
assert torch.allclose(norms, torch.ones(3), atol=1e-5)
print("✅ L2 normalization check passed")

## 4. ✏️ Implement `triplet_loss(anchor, positive, negative, margin=0.2)`

**Signature**:
```python
def triplet_loss(anchor: torch.Tensor, positive: torch.Tensor,
                 negative: torch.Tensor, margin: float = 0.2) -> torch.Tensor:
    # anchor/positive/negative: (B, embed_dim) L2-normalized embeddings
    # returns: scalar Tensor, the batch-mean loss
```

**3 steps to implement**:

| Step | Operation | PyTorch tool |
|---|---|---|
| 1 | Compute the Euclidean distances | `d_pos = (anchor - positive).norm(dim=1)` |
| 2 | `loss_per_sample = max(d_pos - d_neg + margin, 0)` | `torch.clamp(..., min=0)` |
| 3 | Return the batch mean | `.mean()` |

**Acceptance criteria**:
- `anchor==positive, d(a,n)=2.0, margin=0.2` → loss ≈ 0 (the positive is already closer than the negative by margin)
- `d(a,p)=1.5, d(a,n)=1.0, margin=0.5` → loss = max(1.5-1.0+0.5, 0) = 1.0
- The output is a scalar Tensor (shape=())

In [ ]:
def triplet_loss(anchor, positive, negative, margin=0.2):
    """
    Triplet loss for contrastive learning.
    Args:
        anchor, positive, negative: torch.Tensor, shape (B, d)
        margin: float, safety margin
    Returns:
        scalar Tensor
    """
    # ✏️ TODO: implement the three steps d_pos, d_neg, loss in order (delete the raise line when done)
    raise NotImplementedError("TODO: compute d_pos, d_neg, loss")
    # d_pos = torch.norm(anchor - positive, dim=-1)

    # ✏️ TODO: compute d_neg = L2(anchor, negative), one distance per row
    # d_neg = torch.norm(anchor - negative, dim=-1)

    # ✏️ TODO: loss = clamp(d_pos - d_neg + margin, min=0).mean()
    # loss = torch.clamp(d_pos - d_neg + margin, min=0.0).mean()

    return loss

In [ ]:
# Check 1: anchor==positive, negative far away → loss ≈ 0
a = torch.zeros(4, 8)
p = torch.zeros(4, 8)
n = torch.ones(4, 8) * 10
try:
    _loss1 = triplet_loss(a, p, n)
except (NotImplementedError, TypeError):
    _loss1 = None
if _loss1 is None:
    print('⬜ Implement the three TODO items in triplet_loss first')
else:
    val = _loss1.item()
    assert val < 0.01, f"expected loss < 0.01, got {val:.4f}"
    print(f"✅ Check 1 passed: loss = {val:.6f} (should be close to 0)")

    # Check 2: anchor==negative, positive far away → loss ≈ d_pos + margin
    a2 = torch.zeros(4, 8)
    p2 = torch.ones(4, 8) * 10
    n2 = torch.zeros(4, 8)
    _loss2 = triplet_loss(a2, p2, n2)
    if _loss2 is None:
        print('⬜ Implement triplet_loss first')
    else:
        val2 = _loss2.item()
        d_pos_expected = (8 ** 0.5) * 10   # ||ones*10|| for dim=8
        expected = d_pos_expected + 0.2
        assert abs(val2 - expected) < 0.01, f"expected {expected:.4f}, got {val2:.4f}"
        print(f"✅ Check 2 passed: loss = {val2:.4f} (should be ≈ {expected:.4f})")


## 5. Parameter experiment: embedding clusters for three simulated kinds of music

**Setup**: use randomly generated mel features to simulate 3 styles (rock / jazz / classical), 20 songs each, trained for 50 epochs.

**Key parameters**:
- `margin=0.2`: too small → negatives are allowed to stay too close; too large → the loss can never reach 0, so every triplet keeps pushing and training struggles to converge
- `embed_dim=128`: dropping it to 16 can give more visible clusters (the curse of dimensionality bites less)
- `lr=1e-3`: if the convergence curve jitters, lower it to 1e-4

**What to expect**: after training, the t-SNE plot should show three clusters, with the distance between clusters clearly larger than the distance within them.

## 🎯 The payoff later

The **embedding** you've just worked out yourself today (map an object to a vector, and let geometric distance stand for similarity of meaning) will show up again in **L80 / L83 / L88** — L80 takes these very vectors and does nearest-neighbor retrieval; the Transformer in L83 turns every token into an embedding too; and L88's TF-IDF retrieval is its "no neural network" cousin. **"Let the direction of a vector carry meaning" is one single idea that runs through all of modern AI**, and music is just its first practice ground.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ---------- simulated data ----------
torch.manual_seed(42)
np.random.seed(42)

n_mels, T, embed_dim = 64, 50, 128
n_classes, n_per_class = 3, 20
# each class has a different center, so the CNN has a difference to learn
class_centers = [
    torch.randn(1, 1, T, n_mels) * 2 + i * 3
    for i in range(n_classes)
]

def make_mel(class_id):
    """Generate mel features belonging to class_id (with a little noise)."""
    return class_centers[class_id] + torch.randn(1, 1, T, n_mels) * 0.3

all_mels = []  # (class_id, mel_tensor)
for c in range(n_classes):
    for _ in range(n_per_class):
        all_mels.append((c, make_mel(c)))

# ---------- model & optimizer ----------
encoder = MusicEncoder(n_mels=n_mels, embed_dim=embed_dim)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

# ---------- stub check before training ----------
_probe_a = l2_normalize(encoder(make_mel(0)))
_probe_p = l2_normalize(encoder(make_mel(0)))
_probe_n = l2_normalize(encoder(make_mel(1)))
try:
    _probe_loss = triplet_loss(_probe_a, _probe_p, _probe_n)
except (NotImplementedError, TypeError):
    _probe_loss = None

if _probe_loss is None:
    print('⬜ Implement triplet_loss first, then run the training loop')
else:
    # ---------- train for 50 epochs ----------
    losses = []
    for epoch in range(50):
        epoch_loss = 0.0
        indices = torch.randperm(len(all_mels))
        for idx in indices:
            c_a, mel_a = all_mels[idx]
            # positive: another random song of the same class, but never the anchor itself
            pos_idx = torch.randint(n_per_class, (1,)).item()
            while c_a * n_per_class + pos_idx == idx.item():
                pos_idx = torch.randint(n_per_class, (1,)).item()
            _, mel_p = all_mels[c_a * n_per_class + pos_idx]
            # negative: a random song from a different class
            neg_class = (c_a + torch.randint(1, n_classes, (1,)).item()) % n_classes
            neg_idx = torch.randint(n_per_class, (1,)).item()
            _, mel_n = all_mels[neg_class * n_per_class + neg_idx]

            a = l2_normalize(encoder(mel_a))
            p = l2_normalize(encoder(mel_p))
            n_ = l2_normalize(encoder(mel_n))

            loss = triplet_loss(a, p, n_, margin=0.2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        losses.append(epoch_loss / len(all_mels))
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/50  loss={losses[-1]:.4f}")

    # ---------- t-SNE visualization ----------
    encoder.eval()
    embeds, labels = [], []
    with torch.no_grad():
        for c, mel in all_mels:
            e = l2_normalize(encoder(mel)).squeeze(0).numpy()
            embeds.append(e)
            labels.append(c)
    embeds = np.array(embeds)

    try:
        from sklearn.manifold import TSNE
        import matplotlib.pyplot as plt

        tsne = TSNE(n_components=2, perplexity=10, random_state=42)
        proj = tsne.fit_transform(embeds)

        colors = ['#e74c3c', '#3498db', '#2ecc71']
        names  = ['Rock', 'Jazz', 'Classical']
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # t-SNE cluster plot
        for c in range(n_classes):
            mask = np.array(labels) == c
            axes[0].scatter(proj[mask, 0], proj[mask, 1],
                            c=colors[c], label=names[c], s=60, alpha=0.8)
        axes[0].set_title("t-SNE: embedding clusters for three kinds of music")
        axes[0].legend()

        # training loss curve
        axes[1].plot(losses, color='#8e44ad')
        axes[1].set_title("Training loss curve (50 epochs)")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Triplet Loss")

        plt.tight_layout()
        plt.show()
        print("✅ Visualization done")
    except ImportError:
        print("sklearn / matplotlib not installed, skipping the visualization")
        print(f"Final loss: {losses[-1]:.4f}")

## Wrap-up

In this section you implemented `triplet_loss`, which uses L2 distance to pull vectors of the same song together and push vectors of different songs apart, and used `MusicEncoder` (CNN + AdaptiveAvgPool) to compress variable-length mel features into a fixed 128-dimensional vector. Passing the output through `l2_normalize` puts it on the unit sphere, so cosine similarity becomes a plain dot product, saving computation for the FAISS retrieval later. Section 6 introduced the **NT-Xent loss** (the SimCLR variant): it treats every other sample in the batch as a negative, packs in far more information than the triplet loss, and is the mainstream choice for industrial-scale contrastive learning. Section 7 showed the technical path from here to **CLAP** (cross-modal text–audio contrastive learning) and MusicGen/Suno — the encoder in this section is an early sketch of their audio side. This encoder forms the semantic backbone of `aurora.music` — the next section, **L80**, builds a nearest-neighbor retrieval index on these vectors and "finds similar songs" in milliseconds.

## 6. NT-Xent loss (SimCLR): in-batch contrast, stronger than triplets

The triplet loss looks at only one negative per step; **NT-Xent** (Normalized Temperature-Scaled Cross-Entropy) treats every other sample in the batch as a negative, so each step carries far more information — the number of negatives grows with the batch size.

Let the batch size be B, with each `z1[i]` paired with the positive `z2[i]` (two augmented clips of the same song):

```
L_i = -log [ exp(sim(z1_i, z2_i) / τ) / Σ_{k≠i} exp(sim(z1_i, z_k) / τ) ]
```

`τ` (temperature) is the key hyperparameter (typically 0.07–0.2): the smaller it is, the harder it punishes hard negatives and the more concentrated the gradient. This is the core loss of **SimCLR**, and also the mathematical foundation of **CLAP** (Contrastive Language-Audio Pretraining).

**Hard negative mining**: NT-Xent has a built-in hard-negative effect — the more similar a negative is in the denominator Σ, the larger its gradient weight. Explicit enlargement of the negative pool (MoCo style) uses a bigger batch or a memory bank; the practical effect on aurora.music: going from batch size 64 → 128 noticeably sharpens how well the embeddings tell things apart.

In [ ]:
import torch
import torch.nn.functional as F

def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """NT-Xent / SimCLR contrastive loss.

    z1, z2 : shape (B, d); each pair (z1[i], z2[i]) is two augmented clips of the same song (a positive pair).
    Both should already be L2-normalized; the other 2B-2 samples are negatives.
    """
    B = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)                 # (2B, d)
    sim = (z @ z.T) / temperature                   # (2B, 2B)
    sim.fill_diagonal_(float('-inf'))               # mask out self-similarity (never contrast a sample with itself)
    # the positive of z1[i] is z2[i] (index B+i); the positive of z2[i] is z1[i] (index i)
    labels = torch.cat([torch.arange(B, 2 * B),
                        torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, labels)


# ── Check 1: positives nearly identical → loss should approach its theoretical floor ─────────────────────────────────
B, d = 8, 64
torch.manual_seed(0)
z1 = F.normalize(torch.randn(B, d), dim=1)
z2 = F.normalize(z1 + torch.randn(B, d) * 0.01, dim=1)   # small perturbation
loss_good = nt_xent_loss(z1, z2, temperature=0.07)

# ── Check 2: fully random (positives indistinguishable from negatives) → loss far above the identical case ──────────────────
# The chance level log(2B-1) applies only at τ=1 (similarities ≈ 0, so it degenerates to uniform classification);
# the low temperature τ=0.07 amplifies the logits, pushing the random loss well above that level — but the idea is the same: random → high loss.
z2_rand = F.normalize(torch.randn(B, d), dim=1)
loss_rand      = nt_xent_loss(z1, z2_rand, temperature=0.07)
loss_rand_tau1 = nt_xent_loss(z1, z2_rand, temperature=1.0)
upper = torch.log(torch.tensor(2.0 * B - 1))

print(f"positives ≈ identical loss = {loss_good.item():.4f}  (expected low)")
print(f"fully random loss(τ=0.07) = {loss_rand.item():.4f}  (low temperature amplifies it, above the chance level)")
print(f"fully random loss(τ=1.0)  = {loss_rand_tau1.item():.4f}  (expected ≈ log({2*B-1})={upper:.2f})")
assert loss_good < loss_rand, "loss with similar positives should be lower than random"

# ── Temperature sensitivity experiment ────────────────────────────────────────────────────────────
print("\nEffect of temperature on the loss (near-identical positives, batch=16):")
B2, d2 = 16, 64
z1b = F.normalize(torch.randn(B2, d2), dim=1)
z2b = F.normalize(z1b + torch.randn_like(z1b) * 0.05, dim=1)
for tau in [0.05, 0.1, 0.2, 0.5]:
    l = nt_xent_loss(z1b, z2b, temperature=tau).item()
    print(f"  τ={tau:.2f}  loss={l:.4f}")

print("\n✅ NT-Xent checks passed")

## 7. CLAP and the road to MusicGen/Suno

### CLAP: cross-modal NT-Xent

**CLAP** (Contrastive Language-Audio Pretraining) extends NT-Xent across modalities, to **text–audio**:

```
loss = NT-Xent(audio_encoder(audio), text_encoder(text label), τ=0.07)
```

After training, the text embedding for "upbeat jazz piano" lands closest to the embeddings of real jazz piano audio.
LAION's `larger_clap_general` is available directly on HuggingFace (about 900 MB).

### The three core layers of MusicGen (the details of systems like Suno are not public)

```
[text description]
    ↓  T5 / CLAP text encoder → text embedding
[Transformer Decoder]  ← conditioned via cross-attention
    ↓  autoregressively predicts tokens
[EnCodec token sequence]  (50 frames/s × 4 codebooks)
    ↓  EnCodec decoder
[32 kHz audio waveform]
```

aurora.music's `MusicEncoder + NT-Xent` is an early sketch of the **audio side** of the first layer;
once you understand this logic, you understand the overall MusicGen architecture — the only differences are scale (millions of hours of training data) and the text hook-up (T5/CLAP vs this section's lack of text conditioning).

→ **Next step**: open L80 · Vector similarity search, and use cosine similarity to find the top-k most similar tracks in embedding space, checking how well the triplet training worked.

## ✏️ Whiteboard challenge: contrastive learning by hand (target: 10 minutes)

Cover the screen and work it out on paper:

**Question 1**: What does the margin do in the triplet loss? What happens if margin=0?
(The margin demands that the positive be closer than the negative by at least the margin; with margin=0 the model can map every vector to the same point and get loss=0, losing all power to tell things apart)

**Question 2**: If d(a,p)=0.3, d(a,n)=0.8, margin=0.2, what is the triplet loss?
(max(0.3-0.8+0.2, 0) = max(-0.3, 0) = 0.0; the positive is already close enough, the gradient is 0)

**Question 3**: If d(a,p)=0.9, d(a,n)=0.5, margin=0.5, what is the triplet loss?
(max(0.9-0.5+0.5, 0) = max(0.9, 0) = 0.9; the negative is too close and needs pushing away)

**Question 4**: After L2 normalization (v / ||v||), what geometric shape do all the vectors lie on? How are Euclidean distance and cosine similarity related?
(The unit hypersphere; after L2 normalization ||a-b||² = 2(1-cos_sim), so Euclidean distance and cosine similarity correspond monotonically and can be used interchangeably)

**Question 5**: What is the main advantage of the NT-Xent loss over the triplet loss?
(Each step uses the negatives from the whole batch — the other B-1 songs (2B-2 negative clips in the two-view setup of section 6) — while the triplet loss looks at only 1 negative at a time; the bigger the batch, the harder the negatives and the stronger the contrast)

When you've finished the derivations, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import torch
import numpy as np

# Q1: demo of collapse with margin=0
a = torch.zeros(4, 8)   # anchor
p = torch.zeros(4, 8)   # positive (the same point)
n = torch.zeros(4, 8)   # negative (also the same point!)
try:
    loss_zero_margin = triplet_loss(a, p, n, margin=0.0)
    assert abs(loss_zero_margin.item()) < 1e-6, "loss should be 0 when everything is identical"
    print(f"Q1 ✅  margin=0, all vectors identical → loss={loss_zero_margin.item():.4f} (the model can cheat: map everything to one point)")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  the margin prevents cheating: without it, mapping everything to one point gives loss=0 and the gradient vanishes (will be verified once implemented)")

# Q2: d(a,p)=0.3, d(a,n)=0.8, margin=0.2 → loss=0
try:
    embed_dim = 8
    # build vectors with the given distances
    a2 = torch.zeros(1, embed_dim)
    p2 = torch.zeros(1, embed_dim); p2[0, 0] = 0.3  # d(a,p)=0.3
    n2 = torch.zeros(1, embed_dim); n2[0, 0] = 0.8  # d(a,n)=0.8
    loss_q2 = triplet_loss(a2, p2, n2, margin=0.2)
    expected_q2 = max(0.3 - 0.8 + 0.2, 0)
    assert abs(loss_q2.item() - expected_q2) < 1e-4, f"loss should be {expected_q2:.1f}, got {loss_q2.item():.4f}"
    print(f"Q2 ✅  d(a,p)=0.3,d(a,n)=0.8,margin=0.2: loss=max(0.3-0.8+0.2,0)={expected_q2:.1f} (the positive is already close enough)")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  max(0.3-0.8+0.2,0)=max(-0.3,0)=0.0 (the positive is already closer than the negative by the margin; no gradient) (to be implemented)")

# Q3: d(a,p)=0.9, d(a,n)=0.5, margin=0.5 → loss=0.9
try:
    a3 = torch.zeros(1, embed_dim)
    p3 = torch.zeros(1, embed_dim); p3[0, 0] = 0.9
    n3 = torch.zeros(1, embed_dim); n3[0, 0] = 0.5
    loss_q3 = triplet_loss(a3, p3, n3, margin=0.5)
    expected_q3 = max(0.9 - 0.5 + 0.5, 0)
    assert abs(loss_q3.item() - expected_q3) < 1e-4, f"loss should be {expected_q3:.1f}, got {loss_q3.item():.4f}"
    print(f"Q3 ✅  d(a,p)=0.9,d(a,n)=0.5,margin=0.5: loss=max(0.9-0.5+0.5,0)={expected_q3:.1f} (the negative is too close!)")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  max(0.9-0.5+0.5,0)=0.9 (the negative is closer than the positive and needs pushing away) (to be implemented)")

# Q4: how L2 normalization relates to cosine similarity
v1 = torch.tensor([[3.0, 4.0]])  # ||v1||=5
v2 = torch.tensor([[0.0, 1.0]])  # ||v2||=1
v1_norm = v1 / v1.norm()
v2_norm = v2 / v2.norm()
cos_sim = (v1_norm * v2_norm).sum().item()
euclidean_sq = ((v1_norm - v2_norm) ** 2).sum().item()
expected_sq = 2 * (1 - cos_sim)
assert abs(euclidean_sq - expected_sq) < 1e-5, f"||a-b||²=2(1-cos) should hold"
print(f"Q4 ✅  after L2 normalization: ||a-b||²={euclidean_sq:.4f}, 2(1-cos)={expected_sq:.4f} (identical; Euclidean ↔ cosine are interchangeable)")

# Q5: number of negatives in NT-Xent
batch_size = 32
n_negatives_triplet = 1   # Triplet: 1 negative per step
n_negatives_ntxent = batch_size - 1  # NT-Xent: B-1 negatives
print(f"Q5 ✅  batch={batch_size}: Triplet={n_negatives_triplet} negative vs NT-Xent={n_negatives_ntxent} (more → a stronger contrastive signal)")
print("\n🎉 Contrastive-learning whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l79_review = {
    "contrastive_learning_idea":  None,  # understand contrastive learning: similar close, different apart, using triplets? True/False
    "margin_role_understood":     None,  # understand that the margin prevents cheating: without it the model can map every vector to one point? True/False
    "triplet_loss_impl":          None,  # triplet_loss() implemented correctly, all 3 acceptance criteria pass? True/False
    "l2_norm_geometry":           None,  # understand L2 normalization → unit sphere, ||a-b||²=2(1-cos_sim)? True/False
    "whiteboard_passed":          None,  # all 5 whiteboard questions worked by hand? True/False
}

unfilled = [k for k, v in l79_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l79_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L79 complete! On to L80: vector similarity search')

---

→ **Next lesson** · [L80 · Vector similarity search](L80_similarity.ipynb)

> Next lesson covers **vector similarity search**: cosine vs dot product vs L2, a pure-NumPy k-NN.